In [1]:
from lightgbm import LGBMClassifier
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time

warnings.filterwarnings('ignore')

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from cargar_datos import cargar_dataset_recontacto

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score,
    precision_score, recall_score, f1_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
print("✅ Todas las librerías cargadas")

✅ Todas las librerías cargadas


In [2]:
df = cargar_dataset_recontacto()

# Fuera del modelo: identificador, fecha de control y el target
columnas_a_eliminar = ['id_interaccion', 'fecha_contacto', 'es_recontacto']
X = df.drop(columns=columnas_a_eliminar, errors='ignore')
y = df['es_recontacto']

X = X.astype({col: 'int8' for col in X.select_dtypes(include=['bool']).columns})
X = X.fillna(0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Dataset listo: {X_train.shape[0]} train, {X_test.shape[0]} test")
print(f"📊 Features: {X.shape[1]}")

📊 CARGANDO DATASET DE RECONTACTO (KPI 1)
✅ Conexión exitosa a SQL Server
✅ Consulta ejecutada: 11,436 registros, 34 columnas

📏 Dimensiones: 11,436 filas × 34 columnas

🎯 Distribución del target:
es_recontacto
0    6687
1    4749
Name: count, dtype: int64

📈 Tasa de recontacto: 41.53%
✅ Dataset listo: 9148 train, 2288 test
📊 Features: 31


In [3]:
# Ratio de desbalance
ratio = (y_train == 0).sum() / (y_train == 1).sum()

# Diccionario con TODOS los modelos
modelos = {
    '1. Regresión Logística': LogisticRegression(
        max_iter=1000,
        class_weight='balanced',  # Maneja desbalance
        random_state=42
    ),
    
    '2. Árbol de Decisión': DecisionTreeClassifier(
        max_depth=8,
        min_samples_split=20,
        min_samples_leaf=10,
        class_weight='balanced',
        random_state=42
    ),
    
    '3. Random Forest': RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_split=10,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ),
    
    '4. XGBoost': XGBClassifier(
        n_estimators=600,
        max_depth=5,
        learning_rate=0.05,
        scale_pos_weight=ratio,
        random_state=42,
        eval_metric='auc',
        verbosity=0
    ),
    
    '5. LightGBM': LGBMClassifier(
        n_estimators=600,
        max_depth=5,
        learning_rate=0.05,
        scale_pos_weight=ratio,
        random_state=42,
        verbose=-1
    ),
    
    '6. KNN (k=10)': KNeighborsClassifier(
        n_neighbors=10,
        weights='distance',  # Pesa más a los más cercanos
        n_jobs=-1
    ),
    
    '7. Naive Bayes': GaussianNB(),
    
    '8. SVM (RBF)': SVC(
        kernel='rbf',
        probability=True,
        class_weight='balanced',
        random_state=42
    ),
}

print(f"✅ {len(modelos)} modelos configurados")

✅ 8 modelos configurados


In [4]:
resultados = []

print("=" * 80)
print("🤖 ENTRENANDO Y EVALUANDO TODOS LOS MODELOS")
print("=" * 80)

for nombre, modelo in modelos.items():
    print(f"\n🔄 {nombre}...")
    inicio = time.time()
    
    try:
        # Algunos modelos necesitan datos escalados
        if nombre in ['1. Regresión Logística', '6. KNN (k=10)', '8. SVM (RBF)']:
            X_tr, X_te = X_train_scaled, X_test_scaled
        else:
            X_tr, X_te = X_train, X_test
        
        # Entrenar
        modelo.fit(X_tr, y_train)
        
        # Predecir
        y_pred = modelo.predict(X_te)
        y_pred_proba = modelo.predict_proba(X_te)[:, 1]
        
        # Métricas
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_pred_proba)
        
        # Cross-validation (5-fold)
        cv_scores = cross_val_score(modelo, X_tr, y_train, cv=5, scoring='roc_auc')
        cv_mean = cv_scores.mean()
        cv_std = cv_scores.std()
        
        # Tiempo
        tiempo = time.time() - inicio
        
        # Guardar resultados
        resultados.append({
            'Modelo': nombre,
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1-Score': f1,
            'ROC-AUC': auc,
            'CV ROC-AUC': cv_mean,
            'CV Std': cv_std,
            'Tiempo (s)': tiempo
        })
        
        print(f"   ✅ Accuracy: {accuracy:.3f} | AUC: {auc:.3f} | CV-AUC: {cv_mean:.3f}±{cv_std:.3f}")
        print(f"   ⏱️ Tiempo: {tiempo:.2f}s")
        
    except Exception as e:
        print(f"   ❌ Error: {e}")
        resultados.append({
            'Modelo': nombre,
            'Accuracy': 0, 'Precision': 0, 'Recall': 0, 'F1-Score': 0,
            'ROC-AUC': 0, 'CV ROC-AUC': 0, 'CV Std': 0, 'Tiempo (s)': 0
        })

# DataFrame con resultados
df_resultados = pd.DataFrame(resultados).sort_values('ROC-AUC', ascending=False)

🤖 ENTRENANDO Y EVALUANDO TODOS LOS MODELOS

🔄 1. Regresión Logística...
   ✅ Accuracy: 0.804 | AUC: 0.880 | CV-AUC: 0.891±0.011
   ⏱️ Tiempo: 0.32s

🔄 2. Árbol de Decisión...
   ✅ Accuracy: 0.763 | AUC: 0.830 | CV-AUC: 0.838±0.013
   ⏱️ Tiempo: 0.41s

🔄 3. Random Forest...
   ✅ Accuracy: 0.788 | AUC: 0.865 | CV-AUC: 0.873±0.014
   ⏱️ Tiempo: 9.44s

🔄 4. XGBoost...
   ✅ Accuracy: 0.790 | AUC: 0.864 | CV-AUC: 0.872±0.012
   ⏱️ Tiempo: 23.37s

🔄 5. LightGBM...
   ✅ Accuracy: 0.796 | AUC: 0.864 | CV-AUC: 0.870±0.013
   ⏱️ Tiempo: 3.54s

🔄 6. KNN (k=10)...
   ✅ Accuracy: 0.747 | AUC: 0.811 | CV-AUC: 0.808±0.019
   ⏱️ Tiempo: 0.78s

🔄 7. Naive Bayes...
   ✅ Accuracy: 0.782 | AUC: 0.846 | CV-AUC: 0.851±0.014
   ⏱️ Tiempo: 0.15s

🔄 8. SVM (RBF)...
   ✅ Accuracy: 0.793 | AUC: 0.872 | CV-AUC: 0.878±0.013
   ⏱️ Tiempo: 99.84s


In [5]:
print("=" * 100)
print("📊 TABLA COMPARATIVA DE MODELOS")
print("=" * 100)

# Formatear para visualización
df_display = df_resultados.copy()
for col in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'CV ROC-AUC']:
    df_display[col] = df_display[col].apply(lambda x: f"{x:.4f}")
df_display['CV Std'] = df_display['CV Std'].apply(lambda x: f"±{x:.4f}")
df_display['Tiempo (s)'] = df_display['Tiempo (s)'].apply(lambda x: f"{x:.2f}")

print(df_display.to_string(index=False))

# Destacar el mejor modelo
mejor_modelo = df_resultados.iloc[0]['Modelo']
mejor_auc = df_resultados.iloc[0]['ROC-AUC']

print(f"\n🏆 MEJOR MODELO: {mejor_modelo}")
print(f"   ROC-AUC: {mejor_auc:.4f}")

📊 TABLA COMPARATIVA DE MODELOS
                Modelo Accuracy Precision Recall F1-Score ROC-AUC CV ROC-AUC  CV Std Tiempo (s)
1. Regresión Logística   0.8042    0.7413 0.8116   0.7749  0.8803     0.8905 ±0.0106       0.32
          8. SVM (RBF)   0.7933    0.7359 0.7832   0.7588  0.8723     0.8779 ±0.0128      99.84
      3. Random Forest   0.7885    0.7437 0.7484   0.7461  0.8653     0.8733 ±0.0138       9.44
            4. XGBoost   0.7902    0.7374 0.7684   0.7526  0.8643     0.8716 ±0.0124      23.37
           5. LightGBM   0.7959    0.7467 0.7695   0.7579  0.8639     0.8703 ±0.0128       3.54
        7. Naive Bayes   0.7819    0.7844 0.6547   0.7137  0.8457     0.8515 ±0.0137       0.15
  2. Árbol de Decisión   0.7627    0.7149 0.7126   0.7138  0.8300     0.8383 ±0.0130       0.41
         6. KNN (k=10)   0.7474    0.7447 0.5958   0.6620  0.8112     0.8084 ±0.0187       0.78

🏆 MEJOR MODELO: 1. Regresión Logística
   ROC-AUC: 0.8803
